# 01 — Negative transfer on CIFAR-10

**Question.** Under the same MobileNetV3-Small architecture, target examples, split, and epoch budget, does initialization from ImageNet, SVHN, or optionally PathMNIST improve CIFAR-10 accuracy relative to training from scratch?

**H1.** ImageNet initialization may help. **H2.** A mismatched source such as SVHN may help less or produce a negative transfer gap. These are hypotheses, not results. A single negative gap is exploratory; repeat seeds, budgets, and learning rates before a general claim.

Run from the repository root with the project environment. This notebook has no saved outputs and does not start training by default. Historical P1–P5 runs stay under their original `runs/` paths; new results go under `runs/notebooks/`.

## Protocol and mathematical definition

The official CIFAR-10 training partition supplies 2,000 balanced training examples and 5,000 disjoint balanced validation examples using the existing seed-42 split. The latter is split into 2,500 **selection** and 2,500 **guard** examples. This notebook uses selection accuracy to save checkpoints and compare initializations. Notebook 02 opens the guard set once for its final scratch fallback. The official CIFAR-10 test partition is reserved for a final evaluation after the rule and policy are fixed.

For source \(s\) and epoch \(e\), transfer advantage is \(A_s(e)=100[a_s(e)-a_{scratch}(e)]\) percentage points on the same selection images. The final transfer gap uses each candidate's best selection checkpoint, and is explicitly a selection-set observation. Source accuracy, source training time, target training time, and target accuracy must all be reported separately.

In [ ]:
from pathlib import Path
from copy import deepcopy
import csv, json, time, random, platform
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import CIFAR10
import matplotlib.pyplot as plt

from experiments.baselines.train import run_epoch, save_checkpoint, seed_everything
from experiments.negative_transfer.train_target import load_svhn_model
from src.datasets.cifar10 import (balanced_split_indices, get_cifar10_train_transform,
                                  get_cifar10_eval_transform)
from src.guard.decision import split_validation_indices
from src.models.mobilenet import get_model

SEED = 42
TRAIN_SIZE, VALIDATION_SIZE = 2000, 5000
BATCH_SIZE, EPOCHS = 16, 30
LR = {"scratch": 1e-3, "imagenet": 1e-4, "svhn": 1e-4, "pathmnist": 1e-4}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUN_TRAIN = False                 # Explicit opt-in; 30 epochs can take hours.
RUN_SVHN_PRETRAIN = False
RUN_PATHMNIST_PRETRAIN = False
RUN_LR_SWEEP = False
SVHN_CHECKPOINT = None            # e.g. Path("runs/source_svhn/.../best_model.pth")
PATHMNIST_CHECKPOINT = None
ROOT = Path("runs/notebooks/negative_transfer")
seed_everything(SEED)
print("device:", DEVICE)

## Fixed target data

We reuse the P1 balanced index generator and preprocessing. Training augmentation is applied only to training images. Selection and guard use deterministic evaluation transforms. The guard loader is constructed for a split check but is not used for model fitting, checkpoint selection, or transfer-gap plots.

In [ ]:
train_data = CIFAR10("data/raw", train=True, download=True,
                     transform=get_cifar10_train_transform())
eval_data = CIFAR10("data/raw", train=True, download=True,
                    transform=get_cifar10_eval_transform())
train_idx, val_idx = balanced_split_indices(train_data.targets, TRAIN_SIZE, VALIDATION_SIZE, SEED)
selection_idx, guard_idx = split_validation_indices(val_idx, train_data.targets, SEED)
assert len(train_idx) == 2000 and len(selection_idx) == len(guard_idx) == 2500
assert not (set(train_idx) & set(selection_idx) or set(train_idx) & set(guard_idx)
            or set(selection_idx) & set(guard_idx))
assert set(selection_idx) | set(guard_idx) == set(val_idx)
train_loader = DataLoader(Subset(train_data, train_idx), batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0)
selection_loader = DataLoader(Subset(eval_data, selection_idx), batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=0)
print({name: len(indices) for name, indices in
       [("train", train_idx), ("selection", selection_idx), ("guard", guard_idx)]})

## Architecture and source initialization

Every target candidate is TorchVision MobileNetV3-Small with a ten-class head. Scratch starts from random weights. ImageNet starts from the repository's default pretrained weights. An SVHN source checkpoint must be a compatible P3 MobileNetV3-Small checkpoint; loading it replaces only the ten-class output layer. The optional PathMNIST route below uses the same architecture and likewise replaces that layer. Each target run is seeded anew before initialization and training.

In [ ]:
def make_target_model(source, checkpoint=None):
    seed_everything(SEED)
    if source == "scratch":
        return get_model("scratch")
    if source == "imagenet":
        return get_model("finetune")
    if source == "svhn":
        if checkpoint is None:
            raise ValueError("Set SVHN_CHECKPOINT before SVHN target training")
        model, _, _ = load_svhn_model(checkpoint)
        return model
    if source == "pathmnist":
        if checkpoint is None:
            raise ValueError("Set PATHMNIST_CHECKPOINT before PathMNIST target training")
        payload = torch.load(checkpoint, map_location="cpu", weights_only=True)
        if payload.get("mode") != "source_pathmnist":
            raise ValueError("Not a PathMNIST source checkpoint")
        model = get_model("scratch", num_classes=7)
        model.load_state_dict(payload["model_state_dict"], strict=True)
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 10)
        return model
    raise ValueError(source)

assert get_model("scratch").classifier[-1].out_features == 10

## Optional source pretraining

SVHN uses the existing P3 implementation, which selects its source checkpoint on a disjoint SVHN validation subset. Enable the cell only when that source checkpoint is needed. The separate `requirements-svhn.txt` supplies SciPy.

PathMNIST is an optional source ablation and requires `medmnist` installed separately. It uses its official training and validation partitions, RGB conversion, the same 128×128 input and ImageNet normalization, and the same model and cross-entropy loss. Its test partition is not used for source selection. Record the MedMNIST package and dataset version before comparing this source.

In [ ]:
if RUN_SVHN_PRETRAIN:
    from experiments.negative_transfer.pretrain_svhn import main as pretrain_svhn
    source_run = pretrain_svhn(["--epochs", "12", "--train-size", "10000",
                                "--val-size", "2000", "--batch-size", str(BATCH_SIZE),
                                "--seed", str(SEED), "--device", DEVICE.type])
    SVHN_CHECKPOINT = source_run / "best_model.pth"
    print("SVHN source:", SVHN_CHECKPOINT)

In [ ]:
if RUN_PATHMNIST_PRETRAIN:
    import medmnist
    from torchvision import transforms
    from src.datasets.cifar10 import IMAGENET_MEAN, IMAGENET_STD
    transform = transforms.Compose([transforms.Resize((128, 128)),
                                    transforms.ToTensor(),
                                    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
    source_train = medmnist.PathMNIST(split="train", download=True, transform=transform,
                                     as_rgb=True, root="data/raw")
    source_val = medmnist.PathMNIST(split="val", download=True, transform=transform,
                                   as_rgb=True, root="data/raw")
    def source_collate(batch):
        images, labels = zip(*batch)
        return torch.stack(images), torch.tensor([int(np.asarray(y).item()) for y in labels])
    source_train_loader = DataLoader(source_train, batch_size=BATCH_SIZE, shuffle=True,
                                     collate_fn=source_collate, num_workers=0)
    source_val_loader = DataLoader(source_val, batch_size=BATCH_SIZE, shuffle=False,
                                   collate_fn=source_collate, num_workers=0)
    seed_everything(SEED)
    source_model = get_model("scratch", num_classes=7).to(DEVICE)
    optimizer = torch.optim.AdamW(source_model.parameters(), lr=1e-3)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    criterion = nn.CrossEntropyLoss()
    source_dir = ROOT / "source_pathmnist"
    source_dir.mkdir(parents=True, exist_ok=False)
    source_config = dict(dataset="PathMNIST official train/val", architecture="MobileNetV3-Small",
                         seed=SEED, epochs=12, batch_size=BATCH_SIZE, lr=1e-3,
                         medmnist_version=medmnist.__version__, device=DEVICE.type)
    (source_dir / "config.json").write_text(json.dumps(source_config, indent=2))
    best = -1.0
    for epoch in range(1, 13):
        source_model.train()
        run_epoch(source_model, source_train_loader, criterion, DEVICE, optimizer, scaler)
        source_model.eval()
        val_loss, val_acc = run_epoch(source_model, source_val_loader, criterion, DEVICE)
        if val_acc > best:
            best = val_acc
            save_checkpoint(source_dir / "best_model.pth", source_model, epoch,
                            val_loss, val_acc, "source_pathmnist")
    PATHMNIST_CHECKPOINT = source_dir / "best_model.pth"
    print("PathMNIST source:", PATHMNIST_CHECKPOINT)

## Target training and cost

The short loop below retains the verified P2 `run_epoch` numerical path (cross-entropy, AdamW, CUDA autocast) and P2 checkpoint format. The selection set chooses the best epoch. New configuration and metrics files live only under `runs/notebooks/negative_transfer`. For comparison, use the same epoch budget and target split for all sources; a missing source checkpoint excludes that source rather than silently substituting another initialization.

In [ ]:
def train_target(source, checkpoint=None, lr=None):
    lr = LR[source] if lr is None else lr
    seed_everything(SEED)
    model = make_target_model(source, checkpoint).to(DEVICE)
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=lr)
    criterion = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    run_dir = ROOT / f"{source}_seed{SEED}_lr{lr:g}"
    run_dir.mkdir(parents=True, exist_ok=False)
    config = dict(source=source, source_checkpoint=str(checkpoint) if checkpoint else None,
                  architecture="MobileNetV3-Small", dataset="CIFAR-10", seed=SEED,
                  train_size=TRAIN_SIZE, selection_size=len(selection_idx),
                  guard_size=len(guard_idx), batch_size=BATCH_SIZE, epochs=EPOCHS,
                  lr=lr, device=DEVICE.type, torch_version=torch.__version__,
                  torchvision_version=__import__("torchvision").__version__,
                  python_version=platform.python_version(),
                  train_indices=train_idx, selection_indices=selection_idx,
                  guard_indices=guard_idx, test_use="reserved")
    (run_dir / "config.json").write_text(json.dumps(config, indent=2) + "\n")
    rows, best_accuracy, best_epoch = [], -1.0, None
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats(DEVICE)
    start = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss, train_accuracy = run_epoch(model, train_loader, criterion, DEVICE,
                                               optimizer, scaler)
        model.eval()
        val_loss, val_accuracy = run_epoch(model, selection_loader, criterion, DEVICE)
        rows.append(dict(epoch=epoch, train_loss=train_loss, train_accuracy=train_accuracy,
                         selection_loss=val_loss, selection_accuracy=val_accuracy,
                         elapsed_seconds=time.perf_counter() - start))
        if val_accuracy > best_accuracy:
            best_accuracy, best_epoch = val_accuracy, epoch
            save_checkpoint(run_dir / "best_model.pth", model, epoch, val_loss,
                            val_accuracy, source)
    with (run_dir / "metrics.csv").open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=rows[0])
        writer.writeheader()
        writer.writerows(rows)
    summary = dict(source=source, best_selection_accuracy=best_accuracy,
                   best_epoch=best_epoch, target_training_seconds=time.perf_counter() - start,
                   gpu_peak_bytes=(torch.cuda.max_memory_allocated(DEVICE)
                                   if DEVICE.type == "cuda" else None))
    (run_dir / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
    return run_dir

TARGET_RUNS = {}
if RUN_TRAIN:
    for source, checkpoint in [("scratch", None), ("imagenet", None),
                               ("svhn", SVHN_CHECKPOINT), ("pathmnist", PATHMNIST_CHECKPOINT)]:
        if source in ("scratch", "imagenet") or checkpoint is not None:
            TARGET_RUNS[source] = train_target(source, checkpoint)
    print(TARGET_RUNS)

## Scratch convergence and learning-rate sensitivity

A negative gap is uninterpretable if scratch was undertrained. Run at least three prespecified scratch rates (for example 3e-4, 1e-3, 3e-3) with the same budget and inspect the late training/selection curves. If scratch is still improving, increase the budget for **all** target candidates and repeat. Do not select a rate from the guard or test set.

In [ ]:
LR_SWEEP_RUNS = {}
if RUN_LR_SWEEP:
    for rate in (3e-4, 1e-3, 3e-3):
        LR_SWEEP_RUNS[rate] = train_target("scratch", lr=rate)
    print(LR_SWEEP_RUNS)

## Curves, summary table, and interpretation

Point `TARGET_RUNS` at completed **new** runs to inspect them later. Compare only runs with matching indices, seed, epoch budget, and preprocessing. The table reports percentage point gaps against scratch and time for each target fit. A negative value is a candidate observation only. Historical runs remain labeled as P1–P5 and are not mixed into this table.

In [ ]:
def read_run(path):
    path = Path(path)
    config = json.loads((path / "config.json").read_text())
    summary = json.loads((path / "summary.json").read_text())
    with (path / "metrics.csv").open(newline="") as handle:
        history = list(csv.DictReader(handle))
    return config, summary, history

if TARGET_RUNS:
    records = {name: read_run(path) for name, path in TARGET_RUNS.items()}
    reference = records["scratch"][0]
    for name, (config, _, _) in records.items():
        for field in ("train_indices", "selection_indices", "guard_indices",
                      "epochs", "batch_size", "seed"):
            assert config[field] == reference[field], (name, field)
    scratch_accuracy = records["scratch"][1]["best_selection_accuracy"]
    table = []
    for name, (_, summary, history) in records.items():
        table.append(dict(source=name,
                          best_selection_accuracy=summary["best_selection_accuracy"],
                          transfer_gap_pp=100*(summary["best_selection_accuracy"]-scratch_accuracy),
                          best_epoch=summary["best_epoch"],
                          target_training_seconds=summary["target_training_seconds"]))
        plt.plot([int(r["epoch"]) for r in history],
                 [float(r["selection_accuracy"]) for r in history], label=name)
    plt.xlabel("Target epoch"); plt.ylabel("Selection accuracy"); plt.legend(); plt.show()
    for row in table:
        print(row)

**Interpretation to fill after runs.** State the selected scratch learning rate, whether its curve converged, the seed-to-seed spread, source and target training costs, and the direction and uncertainty of each gap. Do not describe a single negative gap as a confirmed general effect. Notebook 02 uses the guard set only after its warning and adaptation choices are fixed.